In [1]:
# Import Libraries

import pandas as pd
import numpy as np
import time
from time import perf_counter_ns
from collections import defaultdict
from pathlib import Path
import re
import math
import gc

In [2]:
# User Values

n_rows = 5000 #296168

h_0 = 0.75
h_1 = 20
r_1 = 20
h_2 = 10
r_2 = 20

epsilon = 0.6 # max sweep volumetric over-representation (%)
beta = 1 # max angular deviation from SLERP (degrees)
delta_step = 1 # segmentation step angle resolution (degrees)

In [3]:
# User Value Checks

# Basic checks

if r_2 < r_1:
    raise SystemExit("Invalid C2 geometry: expected r_2 >= r_1.")

if epsilon <= 0.001:
    raise SystemExit("Abnormally low epsilon value, consider a value above 0.001")

# delta_max

h_delta_13_min = 0.8

alpha_a = r_1 / h_1
alpha_b = h_2 / h_1
alpha_c = r_2 / h_1

# Ratio checks

if not (0.2 <= alpha_a <= 3.0):
    raise SystemExit(f"alpha_a = {alpha_a:.6f} is outside the valid range 0.2 <= alpha_a <= 3.0.")

if not (0.2 <= alpha_b <= 6.0):
    raise SystemExit(f"alpha_b = {alpha_b:.6f} is outside the valid range 0.2 <= alpha_b <= 6.0.")

if not (0.2 <= alpha_c <= 6.0):
    raise SystemExit(f"alpha_c = {alpha_c:.6f} is outside the valid range 0.2 <= alpha_c <= 6.0.")

if alpha_c < alpha_a:
    raise SystemExit("Invalid C2 geometry: expected alpha_c >= alpha_a, equivalent to r_2 >= r_1.")

# Geometric angular limits, in radians
delta_max_a = 2.0 * math.atan(alpha_c / (1.0 + alpha_b))

delta_max_b = 2.0 * (
    math.acos(h_delta_13_min / math.sqrt(1.0 + alpha_c**2))
    - math.atan(alpha_c)
)

delta_max_ab = min(delta_max_a, delta_max_b)


def volume_c2_delta(delta):
    """
    Dimensionless V_C2(delta), using h_1 = 1.
    delta is in radians.
    """
    base_volume = math.pi * ((alpha_a**2 / 3.0) + (alpha_c**2 * alpha_b))

    sweep_term = delta * (
        (2.0 / 3.0) * alpha_a
        + alpha_c * (2.0 * alpha_b + alpha_b**2)
        + (2.0 / 3.0) * alpha_c**3
    )

    return base_volume + sweep_term


def volume_c3_delta(delta):
    """
    Dimensionless V_C3(delta), using h_1 = 1.
    delta is in radians.
    """
    c = math.cos(delta / 2.0)
    s = math.sin(delta / 2.0)

    denominator = c - alpha_a * s

    if abs(denominator) < 1e-12:
        return math.inf

    cylinder_term = (
        (alpha_c * c + (1.0 + alpha_b) * s)**2
        * (alpha_b * c + 2.0 * alpha_c * s)
    )

    cone_term = (
        ((alpha_a * c + s)**2 * (c - alpha_c * s)**3)
        / (3.0 * denominator**2)
    )

    return math.pi * (cylinder_term + cone_term)


def epsilon_delta(delta):

    v_c2 = volume_c2_delta(delta)
    v_c3 = volume_c3_delta(delta)

    return (v_c3 - v_c2) / v_c2


# delta_max_c
if epsilon == 0:
    delta_max_c = 0.0

elif epsilon_delta(delta_max_ab) <= epsilon:
    
    delta_max_c = delta_max_ab

else:
    
    lo = 0.0
    hi = delta_max_ab

    for _ in range(80):
        mid = 0.5 * (lo + hi)

        if epsilon_delta(mid) <= epsilon:
            lo = mid
        else:
            hi = mid

    delta_max_c = lo



delta_max = min(delta_max_a, delta_max_b, delta_max_c)
delta_max_deg = math.degrees(delta_max)

print(f"alpha_a     = {alpha_a:.6f}")
print(f"alpha_b     = {alpha_b:.6f}")
print(f"alpha_c     = {alpha_c:.6f}")
#print(f"delta_max_a = {delta_max_a:.6f} rad")
#print(f"delta_max_b = {delta_max_b:.6f} rad")
#print(f"delta_max_c = {delta_max_c:.6f} rad")
print(f"delta_max   = {delta_max:.6f} rad ({delta_max_deg:.6f} deg)")
print(f"2beta       = {np.radians(2 * beta):.6f} rad")
print(f"Tip clearance h0: {h_0:.6f}")

alpha_a     = 1.000000
alpha_b     = 0.500000
alpha_c     = 1.000000
delta_max   = 0.368268 rad (21.100196 deg)
2beta       = 0.034907 rad
Tip clearance h0: 0.750000


In [4]:
# Import Waypoints

folder_path = Path(r"C:\S3_3DP_MotionPlanning\DataSet\Sorce\bunnyHead\waypoint")

def natural_sort_key(path):
    return [
        int(part) if part.isdigit() else part.lower()
        for part in re.split(r"(\d+)", path.name)
    ]

txt_files = sorted(folder_path.glob("*.txt"), key=natural_sort_key)

all_data = []
rows_imported = 0

for txt_file in txt_files:

    if rows_imported >= n_rows:
        break

    data = np.loadtxt(txt_file, dtype=float)
    data = np.atleast_2d(data)

    if data.shape[1] != 6:
        raise ValueError(
            f"Expected 6 columns in {txt_file.name}, but found {data.shape[1]}"
        )

    original_file_rows = data.shape[0]

    remaining_rows = n_rows - rows_imported
    rows_to_take = min(remaining_rows, original_file_rows)

    data = data[:rows_to_take]

    layer_end = np.zeros((rows_to_take, 1), dtype=float)

    # Mark true end of txt file if imported chunk reaches it
    if rows_to_take == original_file_rows:
        layer_end[-1, 0] = 1.0

    data_with_marker = np.hstack((data, layer_end))
    all_data.append(data_with_marker)

    rows_imported += rows_to_take

if not all_data:
    raise ValueError("No data was imported.")

A = np.vstack(all_data)
del all_data
gc.collect()

# Mark the final imported line as an endpoint,
# even if n_rows stops halfway through a txt file.
A[-1, 6] = 1.0

df = pd.DataFrame(
    A,
    columns=["x", "y", "z", "i", "j", "k", "layer_end"]
)

df["layer_end"] = df["layer_end"].astype(np.int8)

A = df.to_numpy(dtype=float)

#print("\nSample rows:")
#print(df.head(10).to_string())

# Add q_1 as the row number
df["q_1"] = np.arange(len(df), dtype=np.int64)

# Add q_2 as integer zeros
df["q_2"] = np.zeros(len(df), dtype=np.int64)

# Create next-line x, y, z values
df["x+"] = df["x"].shift(-1)
df["y+"] = df["y"].shift(-1)
df["z+"] = df["z"].shift(-1)

# If the current row is the end of a layer, leave x+, y+, z+ blank
layer_end_mask = df["layer_end"] == 1

df.loc[layer_end_mask, ["x+", "y+", "z+"]] = np.nan

# Reorder columns so x+, y+, z+ appear after x, y, z
df = df[
    [
        "x", "y", "z",
        "x+", "y+", "z+",
        "i", "j", "k",
        "layer_end",
        "q_1", "q_2"
    ]
]

# Create next-line i, j, k values
df["i+"] = df["i"].shift(-1)
df["j+"] = df["j"].shift(-1)
df["k+"] = df["k"].shift(-1)

df = df[
    [
        "x", "y", "z",
        "x+", "y+", "z+",
        "i", "j", "k",
        "i+", "j+", "k+",
        "layer_end",
        "q_1", "q_2"
    ]
]

# If the current row is the end of a layer, leave i+, j+, k+ blank
layer_end_mask = df["layer_end"] == 1

df.loc[layer_end_mask, ["i+", "j+", "k+"]] = np.nan

# Delete any row where layer_end = 1
df = df[df["layer_end"] != 1].copy()

# Delete the layer_end column
df = df.drop(columns=["layer_end"])

# Optional: reset the dataframe index after deleting rows
df = df.reset_index(drop=True)

A_ref = A.copy()

# Calculate omega angle between [i, j, k] and [i+, j+, k+]

dot_product = (
    df["i"] * df["i+"] +
    df["j"] * df["j+"] +
    df["k"] * df["k+"]
)

# Protect against floating-point rounding errors outside [-1, 1]
dot_product = np.clip(dot_product, -1.0, 1.0)

# Omega in radians
df["omega"] = np.arccos(dot_product)

# Convert back to NumPy array
A = df.to_numpy(dtype=float)

print(f"A shape: {A.shape}")

A shape: (4992, 15)


In [5]:
# Primary Segmentation

columns = [
    "x", "y", "z",
    "x+", "y+", "z+",
    "i", "j", "k",
    "i+", "j+", "k+",
    "q_1", "q_2",
    "omega"
]

df = pd.DataFrame(A, columns=columns)

beta_rad = np.radians(beta)
delta_step_rad = np.radians(delta_step)

def slerp_unit_vectors(u1, u2, t_values, omega):
    """
    SLERP between two unit orientation vectors u1 and u2.

    u1, u2: shape (3,)
    t_values: shape (N,)
    omega: angular separation in radians
    """

    sin_omega = np.sin(omega)

    if abs(sin_omega) < 1e-12:
        # Fallback for extremely small angles.
        u = (1.0 - t_values[:, None]) * u1 + t_values[:, None] * u2
    else:
        w1 = np.sin((1.0 - t_values) * omega) / sin_omega
        w2 = np.sin(t_values * omega) / sin_omega

        u = w1[:, None] * u1 + w2[:, None] * u2

    # Re-normalise to protect against tiny floating-point drift
    norms = np.linalg.norm(u, axis=1)

    if np.any(norms == 0):
        raise ValueError("A zero-length orientation vector was produced during SLERP.")

    u = u / norms[:, None]

    return u


new_rows = []

for _, row in df.iterrows():

    omega = float(row["omega"])

    # Keep invalid or near zero omega rows unchanged
    if not np.isfinite(omega):
        new_rows.append(row.to_dict())
        continue

    # Only split rows where omega > 2*beta
    if omega <= 2.0 * beta_rad:
        row_dict = row.to_dict()
        row_dict["q_2"] = int(row_dict["q_2"])
        new_rows.append(row_dict)
        continue

    # Number of output line segments
    N_p = int(np.ceil((omega - 2.0 * beta_rad) / delta_step_rad))

    if N_p < 1:
        N_p = 1

    T_1 = np.array([row["x"], row["y"], row["z"]], dtype=float)
    T_2 = np.array([row["x+"], row["y+"], row["z+"]], dtype=float)

    u_1 = np.array([row["i"], row["j"], row["k"]], dtype=float)
    u_2 = np.array([row["i+"], row["j+"], row["k+"]], dtype=float)

    # Normalise input vectors defensively
    u_1_norm = np.linalg.norm(u_1)
    u_2_norm = np.linalg.norm(u_2)

    if u_1_norm == 0 or u_2_norm == 0:
        raise ValueError(f"Zero-length unit vector found at q_1 = {row['q_1']}")

    u_1 = u_1 / u_1_norm
    u_2 = u_2 / u_2_norm

    # Define theta values.
    # There are N_p + 1 points, giving N_p output line rows.
    theta_values = np.empty(N_p + 1, dtype=float)

    theta_values[0] = 0.0
    theta_values[-1] = omega

    if N_p > 1:
        m_values = np.arange(1, N_p, dtype=float)

        # As specified:
        # theta_m = beta + m*delta_step
        theta_values[1:-1] = beta_rad + m_values * delta_step_rad

    # Convert angular positions into SLERP parameters
    t_values = theta_values / omega

    # Tip positions along L_1,2
    T_values = T_1 + t_values[:, None] * (T_2 - T_1)

    # Unit orientation vectors using SLERP
    u_values = slerp_unit_vectors(u_1, u_2, t_values, omega)

    original_q_1 = int(row["q_1"])

    # Create N_p output rows
    for m in range(N_p):

        T_start = T_values[m]
        T_end = T_values[m + 1]

        u_start = u_values[m]
        u_end = u_values[m + 1]

        sub_omega = theta_values[m + 1] - theta_values[m]

        new_rows.append({
            "x": T_start[0],
            "y": T_start[1],
            "z": T_start[2],

            "x+": T_end[0],
            "y+": T_end[1],
            "z+": T_end[2],

            "i": u_start[0],
            "j": u_start[1],
            "k": u_start[2],

            "i+": u_end[0],
            "j+": u_end[1],
            "k+": u_end[2],

            "q_1": original_q_1,
            "q_2": int(m),

            "omega": sub_omega
        })


df = pd.DataFrame(new_rows, columns=columns)

df = df.rename(columns={"omega": "delta_seg"})

columns = [
    "x", "y", "z",
    "x+", "y+", "z+",
    "i", "j", "k",
    "i+", "j+", "k+",
    "q_1", "q_2",
    "delta_seg"
]

# Keep q_1 and q_2 as integer-like values in the dataframe
df["q_1"] = df["q_1"].astype(np.int64)
df["q_2"] = df["q_2"].astype(np.int64)

# Convert back to NumPy array
A = df.to_numpy(dtype=float)

del new_rows
gc.collect()

print(f"New A shape: {A.shape}")
print(f"Max delta_seg: {df['delta_seg'].max():.6f} rad ({np.degrees(df['delta_seg'].max()):.6f} deg)")

New A shape: (5884, 15)
Max delta_seg: 0.052079 rad (2.983910 deg)


In [6]:
# Secondary Segmentation

columns = [
    "x", "y", "z",
    "x+", "y+", "z+",
    "i", "j", "k",
    "i+", "j+", "k+",
    "q_1", "q_2",
    "delta_seg"
]

df = pd.DataFrame(A, columns=columns)

# Add q_3 after q_2, initially all integer zeros
df["q_3"] = np.zeros(len(df), dtype=np.int64)

# Reorder columns so q_3 is immediately after q_2
columns_with_q3 = [
    "x", "y", "z",
    "x+", "y+", "z+",
    "i", "j", "k",
    "i+", "j+", "k+",
    "q_1", "q_2", "q_3",
    "delta_seg"
]

df = df[columns_with_q3]

if delta_max <= 0:
    raise ValueError("delta_max must be greater than zero.")

max_delta_seg = df["delta_seg"].max()

def slerp_unit_vectors(u1, u2, t_values, omega):
    """
    SLERP between two unit orientation vectors.

    u1, u2: shape (3,)
    t_values: shape (N,)
    omega: angular separation in radians
    """

    sin_omega = np.sin(omega)

    if abs(sin_omega) < 1e-12:
        # Fallback for very small angles
        u = (1.0 - t_values[:, None]) * u1 + t_values[:, None] * u2
    else:
        w1 = np.sin((1.0 - t_values) * omega) / sin_omega
        w2 = np.sin(t_values * omega) / sin_omega

        u = w1[:, None] * u1 + w2[:, None] * u2

    # Re-normalise to protect against floating-point drift
    norms = np.linalg.norm(u, axis=1)

    if np.any(norms == 0):
        raise ValueError("A zero-length orientation vector was produced during SLERP.")

    u = u / norms[:, None]

    return u


if max_delta_seg > delta_max:

    new_rows = []

    rows_before = len(df)
    rows_to_split = (df["delta_seg"] > delta_max).sum()

    print(f"Rows requiring secondary segmentation: {rows_to_split}")

    for _, row in df.iterrows():

        delta_seg = float(row["delta_seg"])

        # Keep rows unchanged if they do not exceed delta_max
        if delta_seg <= delta_max or not np.isfinite(delta_seg):

            row_dict = row.to_dict()
            row_dict["q_1"] = int(row_dict["q_1"])
            row_dict["q_2"] = int(row_dict["q_2"])
            row_dict["q_3"] = 0

            new_rows.append(row_dict)
            continue

        # Number of secondary orientation sub-segments
        N_s = int(np.ceil(delta_seg / delta_max))

        if N_s < 1:
            N_s = 1

        # Equal new angular segment size
        new_delta_seg = delta_seg / N_s

        u_1 = np.array([row["i"], row["j"], row["k"]], dtype=float)
        u_2 = np.array([row["i+"], row["j+"], row["k+"]], dtype=float)

        # Normalise defensively
        u_1_norm = np.linalg.norm(u_1)
        u_2_norm = np.linalg.norm(u_2)

        if u_1_norm == 0 or u_2_norm == 0:
            raise ValueError(
                f"Zero-length unit vector found at q_1 = {row['q_1']}, q_2 = {row['q_2']}"
            )

        u_1 = u_1 / u_1_norm
        u_2 = u_2 / u_2_norm

        # N_s rows require N_s + 1 orientation vectors
        t_values = np.linspace(0.0, 1.0, N_s + 1)

        u_values = slerp_unit_vectors(
            u1=u_1,
            u2=u_2,
            t_values=t_values,
            omega=delta_seg
        )

        original_q_1 = int(row["q_1"])
        original_q_2 = int(row["q_2"])

        for s in range(N_s):

            u_start = u_values[s]
            u_end = u_values[s + 1]

            new_rows.append({
                # Position values remain unchanged
                "x": row["x"],
                "y": row["y"],
                "z": row["z"],

                "x+": row["x+"],
                "y+": row["y+"],
                "z+": row["z+"],

                # Orientation values are subdivided
                "i": u_start[0],
                "j": u_start[1],
                "k": u_start[2],

                "i+": u_end[0],
                "j+": u_end[1],
                "k+": u_end[2],

                # q_1 and q_2 are maintained
                "q_1": original_q_1,
                "q_2": original_q_2,

                # q_3 is the secondary subdivision counter
                "q_3": int(s),

                # Equal new angular segment size
                "delta_seg": new_delta_seg
            })

    df = pd.DataFrame(new_rows, columns=columns_with_q3)

    df["q_1"] = df["q_1"].astype(np.int64)
    df["q_2"] = df["q_2"].astype(np.int64)
    df["q_3"] = df["q_3"].astype(np.int64)

    print(f"New A shape: {df.shape}")
    print(f"Max delta_seg (radians): {df['delta_seg'].max()}")
    print(f"Max delta_seg (degrees): {np.degrees(df['delta_seg'].max())}")

else:
    print("No eligible rows for secondary segmentation.")
    print(f"New A shape: {df.shape}")

# Convert back to NumPy array
A = df.to_numpy(dtype=float)

No eligible rows for secondary segmentation.
New A shape: (5884, 16)


In [7]:
# Establish u3

# Current expected dataframe structure:
columns = [
    "x", "y", "z",
    "x+", "y+", "z+",
    "i", "j", "k",
    "i+", "j+", "k+",
    "q_1", "q_2", "q_3",
    "delta_seg"
]

# Rebuild df from A if needed
df = pd.DataFrame(A, columns=columns)

# Extract start and end unit orientation vectors
u_1 = df[["i", "j", "k"]].to_numpy(dtype=float)
u_2 = df[["i+", "j+", "k+"]].to_numpy(dtype=float)

# Normalise defensively
u_1_norms = np.linalg.norm(u_1, axis=1)
u_2_norms = np.linalg.norm(u_2, axis=1)

if np.any(u_1_norms == 0):
    raise ValueError("At least one [i, j, k] vector has zero length.")

if np.any(u_2_norms == 0):
    raise ValueError("At least one [i+, j+, k+] vector has zero length.")

u_1 = u_1 / u_1_norms[:, None]
u_2 = u_2 / u_2_norms[:, None]

# Recalculate angular separation directly from the vectors
dot_product = np.sum(u_1 * u_2, axis=1)
dot_product = np.clip(dot_product, -1.0, 1.0)

omega_direct = np.arccos(dot_product)

# Calculate SLERP midpoint at t = 0.5
sin_omega = np.sin(omega_direct)

u_3 = np.empty_like(u_1)

small_angle_mask = np.abs(sin_omega) < 1e-12

# For very small angles, the midpoint is effectively u_1
u_3[small_angle_mask] = u_1[small_angle_mask]

# Normal SLERP midpoint
normal_mask = ~small_angle_mask

weight = np.sin(0.5 * omega_direct[normal_mask]) / sin_omega[normal_mask]

u_3[normal_mask] = (
    weight[:, None] * u_1[normal_mask]
    + weight[:, None] * u_2[normal_mask]
)

# Re-normalise to protect against floating-point drift
u_3_norms = np.linalg.norm(u_3, axis=1)

if np.any(u_3_norms == 0):
    raise ValueError("At least one midpoint vector [i3, j3, k3] has zero length.")

u_3 = u_3 / u_3_norms[:, None]

# Add midpoint orientation columns
df["i3"] = u_3[:, 0]
df["j3"] = u_3[:, 1]
df["k3"] = u_3[:, 2]

# Remove original start/end orientation-vector columns
df = df.drop(columns=["i", "j", "k", "i+", "j+", "k+"])

# Reorder columns
df = df[
    [
        "x", "y", "z",
        "x+", "y+", "z+",
        "i3", "j3", "k3",
        "q_1", "q_2", "q_3",
        "delta_seg"
    ]
]

# Convert back to NumPy array
A = df.to_numpy(dtype=float)

del u_1, u_2, u_3
del u_1_norms, u_2_norms, u_3_norms
del dot_product, omega_direct, sin_omega
gc.collect()

0

In [8]:
# Establish C3

# Current expected dataframe structure
columns = [
    "x", "y", "z",
    "x+", "y+", "z+",
    "i3", "j3", "k3",
    "q_1", "q_2", "q_3",
    "delta_seg"
]

# Rebuild df from A
df = pd.DataFrame(A, columns=columns)

# delta is delta_seg and is already stored in radians
delta = df["delta_seg"].to_numpy(dtype=float)
half_delta = 0.5 * delta

cos_half = np.cos(half_delta)
sin_half = np.sin(half_delta)

# Denominator used in r_3
r3_denominator = h_1 * cos_half - r_1 * sin_half

# Safety check to avoid division by zero or near-zero denominator
if np.any(np.abs(r3_denominator) < 1e-12):
    raise ValueError(
        "At least one r_3 denominator is close to zero. "
        "Check h_1, r_1, and delta_seg values."
    )

# Calculate h3, h4, r3, and r4
df["r3"] = (
    (r_1 * cos_half + h_1 * sin_half)
    *
    ((h_1 * cos_half - r_2 * sin_half) / r3_denominator)
)

df["h3"] = h_1 * cos_half - r_2 * sin_half

df["r4"] = r_2 * cos_half + (h_1 + h_2) * sin_half

df["h4"] = h_2 * cos_half + 2.0 * r_2 * sin_half

# Reorder columns so h3, h4, r3, r4 appear after k3
df = df[
    [
        "x", "y", "z",
        "x+", "y+", "z+",
        "i3", "j3", "k3",
        "h3", "h4", "r3", "r4",
        "q_1", "q_2", "q_3",
        "delta_seg"
    ]
]

# Convert back to NumPy array
A = df.to_numpy(dtype=float)

In [9]:
# Output to CSVs

np.savetxt(
    "A.csv",
    A,
    delimiter=",",
    fmt="%.6f"
)

np.savetxt(
    "A_ref.csv",
    A_ref,
    delimiter=",",
    fmt="%.6f"
)

print(f"CSV Files Saved!")

CSV Files Saved!
